# Basic

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import polars as pl
import pandas as pd
import pickle
import numpy as np
import os
import tqdm

import src.graph_tokenizer_gd_tree_dev.config as config
import src.graph_tokenizer_gd_tree_dev.tokenizer as tokenizer
import src.graph_tokenizer_gd_tree_dev.eval as eval

# cohort + case/control label

`Liste patients totale.xlsx`: `PatID` / `Profil` ("Patient" = case/aneurysm, "Control" = control).
`PatID` is `real_patient_id` -- bridge to `unique_patient_id` (the trajectory join key) via
`patient_ids_and_index_lookup.parquet`, which maps one `unique_patient_id` to a list of
`real_patient_id`s (explode before joining).

In [3]:
df_labels = (
    pl.from_pandas(pd.read_excel(config.IACohort().patient_list_path))
    .rename({"PatID": "real_patient_id"})
    .with_columns((pl.col("Profil") == "Patient").cast(pl.Int8).alias("is_case"))
    .select("real_patient_id", "is_case")
)

df_lookup = (
    pl.read_parquet(config.TimelineData().patient_ids_and_index_lookup)
    .explode("real_patient_id")
    .select("unique_patient_id", "real_patient_id")
)

df_cohort = df_lookup.join(df_labels, on="real_patient_id", how="inner").select("unique_patient_id", "is_case").unique()

print(f"cohort: {df_cohort.shape[0]:,} patients   case: {df_cohort['is_case'].sum():,}   control: {(1 - df_cohort['is_case']).sum():,}")

cohort: 6,108 patients   case: 3,044   control: 3,064


# gender feature

In [4]:
df_gender = (
    pl.read_parquet(config.TimelineData().patient_gender)
    .unique()
    .with_columns((pl.col("gender") == "Female (finding)").cast(pl.Int8).alias("gender_female"))
    .select("unique_patient_id", "gender_female")
)

df_cohort = df_cohort.join(df_gender, on="unique_patient_id", how="left")
n_missing_gender = df_cohort["gender_female"].is_null().sum()
df_cohort = df_cohort.with_columns(pl.col("gender_female").fill_null(0))
print(f"gender missing for {n_missing_gender:,} / {df_cohort.shape[0]:,} patients -- filled with 0")

gender missing for 2 / 6,108 patients -- filled with 0


# trajectories: restrict to cohort + SNOMED, one row per (patient, distinct concept)

In [5]:
df_traj_cohort = (
    pl.read_parquet(config.TimelineData().all_patient_traj)
    .filter(pl.col("mapping_type") == "SNOMED")
    .join(df_cohort.select("unique_patient_id"), on="unique_patient_id", how="inner")
    .select("unique_patient_id", "mapped_code")
    .unique()
)

print(f"{df_traj_cohort.shape[0]:,} distinct (patient, concept) pairs, "
      f"{df_traj_cohort['unique_patient_id'].n_unique():,} patients with >=1 SNOMED concept, "
      f"{df_traj_cohort['mapped_code'].n_unique():,} distinct concepts in cohort")

1,036,980 distinct (patient, concept) pairs, 6,106 patients with >=1 SNOMED concept, 6,002 distinct concepts in cohort


# graph + id_to_label + shared out-adjacency (built once, reused for every method/k)

In [6]:
with open(config.IAProcessedGraph().combined_subgraphs, "rb") as f:
    combined_subgraphs = pickle.load(f)

with open(config.ProcessedGraph().id_to_label, "rb") as f:
    id_to_label = pickle.load(f)

D = config.TokenizerParam().max_dist_candidate
adj = tokenizer.build_out_adjacency(combined_subgraphs)

unique_concepts_in_cohort = df_traj_cohort["mapped_code"].unique().to_list()
print(f"D={D}   unique concepts to tokenize per method/k: {len(unique_concepts_in_cohort):,}")

D=3   unique concepts to tokenize per method/k: 6,002


# method specs: 3 lambdas + 2 baselines, x 3 k's = 15 combos

In [7]:
method_specs = {
    "lam_0.6": config.IACandidateLists().path_greedy_tree + "0.6.parquet",
    "lam_0.8": config.IACandidateLists().path_greedy_tree + "0.8.parquet",
    "lam_1.0": config.IACandidateLists().path_greedy_tree + "1.0.parquet",
    "personalized_pagerank": config.IACandidateLists().personalized_pagerank,
    "discrete_set_cover": config.IACandidateLists().discrete_set_cover,
}

Ks = np.arange(100, 6000, 100)          # same grid as 6.1's selection sweep

combos = [(name, k) for name in method_specs for k in Ks]
print(f"{len(combos)} (method, k) combos")

295 (method, k) combos


# tokenize once per unique concept (not per patient) -- concept -> (tokens found, any-uncovered)

`tokenize_all_rel(c, adj, T, D, id_to_label)` only depends on `(c, T, D)`, never on which patient
`c` came from, so every distinct concept in the cohort is tokenized exactly once per `(method, k)`
instead of once per `(patient, concept)` pair -- collapses ~millions of walks down to a few
thousand.

In [8]:
def concept_to_tokens_and_unk(c, T_set, D):
    tree = tokenizer.tokenize_all_rel(c, adj, T_set, D, id_to_label)
    contexts = list(eval._iter_contexts(tree))
    found = {tok for ctx in contexts for tok, _ in ctx.tokens}
    uncovered = any(ctx.uncovered for ctx in contexts)
    return found, uncovered

# build boolean feature matrix per (method, k) and save

In [9]:
os.makedirs(config.IAFeatures().path, exist_ok=True)

traj_by_patient = df_traj_cohort.group_by("unique_patient_id").agg(pl.col("mapped_code").alias("concepts"))
cohort_pd = df_cohort.to_pandas().set_index("unique_patient_id")

for method_name, k in tqdm.tqdm(combos):
    df_ranked = pl.read_parquet(method_specs[method_name])
    T = df_ranked.head(k)["token"].to_list()
    T_set = set(T)
    tok_index = {t: i for i, t in enumerate(T)}

    concept_lookup = {c: concept_to_tokens_and_unk(c, T_set, D) for c in unique_concepts_in_cohort}

    rows = []
    for pid, concepts in zip(traj_by_patient["unique_patient_id"], traj_by_patient["concepts"]):
        vec = np.zeros(k, dtype=np.int8)
        has_unk = False
        for c in concepts:
            found, uncovered = concept_lookup[c]
            for t in found:
                vec[tok_index[t]] = 1
            has_unk = has_unk or uncovered
        rows.append((pid, has_unk, *vec))

    feat_cols = [f"tok_{i}" for i in range(k)]
    df_feat = (
        pl.DataFrame(rows, schema=["unique_patient_id", "has_unk"] + feat_cols, orient="row")
        .join(df_cohort, on="unique_patient_id", how="left")
    )
    df_feat.write_parquet(f"{config.IAFeatures().path}{method_name}_k{k}.parquet")

print(f"wrote {len(combos)} feature files to {config.IAFeatures().path}")

100%|██████████| 295/295 [39:55<00:00,  8.12s/it]

wrote 295 feature files to /mnt/z/graph_tokenizer_greedy_tree/IA_patient_analysis/features/


# no-tokenizer baseline: raw concept presence/absence, no reduction

The upper-bound/sanity-check reference point -- one boolean column per distinct concept
actually seen in the cohort, no graph walk, no reduction. Same schema as the 15 tokenizer
combos (`has_unk` always `False` here -- there's no coverage notion without a tokenizer) so
it drops into the same downstream classification/clustering code unchanged. Also saves the
column order (`no_tokenizer_vocab.parquet`) since, unlike the tokenizer combos, there's no
ranking parquet to re-derive it from later.

In [11]:
all_concepts = sorted(unique_concepts_in_cohort)
concept_index = {c: i for i, c in enumerate(all_concepts)}
k_full = len(all_concepts)

rows = []
for pid, concepts in zip(traj_by_patient["unique_patient_id"], traj_by_patient["concepts"]):
    vec = np.zeros(k_full, dtype=np.int8)
    for c in concepts:
        vec[concept_index[c]] = 1
    rows.append((pid, False, *vec))

feat_cols = [f"tok_{i}" for i in range(k_full)]
df_feat_full = (
    pl.DataFrame(rows, schema=["unique_patient_id", "has_unk"] + feat_cols, orient="row")
    .join(df_cohort, on="unique_patient_id", how="left")
)
df_feat_full.write_parquet(f"{config.IAFeatures().path}no_tokenizer_kfull.parquet")

pl.DataFrame({"index": range(k_full), "token": all_concepts}).write_parquet(
    f"{config.IAFeatures().path}no_tokenizer_vocab.parquet"
)

print(f"no-tokenizer baseline: {k_full:,} raw concept dimensions (vs. 500-2000 for the reduced methods)")

no-tokenizer baseline: 6,002 raw concept dimensions (vs. 500-2000 for the reduced methods)
